« RGN — Rasyonel Çekim Ağı · COLAB KOŞULARI »

**Soru: ters kare çekim dikkati ve rasyonel MLP ile kurulan decoder (konum RoPE), aynı bütçeyle softmax transformer'a ne kadar yaklaşıyor?**
Model: `rgn/rgn/` (sekerai deposu, kol `BRANCH`); deneyler `rgn/configs/`; veri Drive'dan; çıktılar `Drive/RGN/runs/<koşu>/` (yerelde `G:\Drive'ım\RGN\runs`).
Tasarım dokümanı yerelde (`rgn/docs/`, git dışı).

| hücre | iş |
|---|---|
| 0 HAZIRLIK | Drive, kod (GitHub'dan taze, `COMMIT` yazdırılır), bağımlılık; canlı koşu varken yeniden kurmaz |
| 1 TEST | GPU kapısı; test takımı arka planda (CUDA testleri: flex = dense, γ/β gradyanı) |
| 2 KOŞU | GPU kapısı; deney listesi sırayla arka planda (`R.kuyruk`), her koşu sonunda hız profili |
| 3 NABIZ | işler: durum, adım, kayıp, kalan süre, son değerlendirme |
| 4 SONUÇ | SAYI + METİN: kayıp / bpb / hız / bellek + greedy örnekler, karşılaştırma tablosu |
| 5 HIZ | dikkat çekirdekleri N = 512..4096 (arka planda) |
| X DURDUR | SIGTERM: checkpoint kaydedip çıkar; 2 KOŞU aynı listeyle kaldığı yerden sürer |
| Y DRIVE'DAKİ KAYIT | koşu klasörleri ve dosyaları |
| Z GPU | nvidia-smi; iş yoksa çalışma zamanını kapat |


In [ ]:
# 0 HAZIRLIK  |  CPU  |  tekrar: GÜVENLİ
# Çekirdek düştüyse ÖNCE bu hücre.  Kod git'ten (önce commit + push), veri Drive'dan.
REPO, BRANCH = 'https://github.com/sekerahmet/sekerai.git', 'deneme/rgn-asama0'
import os, sys, importlib
if 'runner' in sys.modules:                                   # canlı iş varken kod değiştirilmez
    _R = sys.modules['runner']
    _live = [n for n, j in _R._jobs().items() if not n.startswith('_') and _R._alive(j['pid'])]
    assert not _live, 'ÇALIŞAN İŞ VAR: %s -- HAZIRLIK gerekmiyor, 3 NABIZ' % _live
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!rm -rf /content/_repo && git clone -q --depth 1 --branch {BRANCH} {REPO} /content/_repo
!python /content/_repo/rgn/colab/setup_colab.py --git {REPO} --branch {BRANCH} --subdir rgn
for _m in [m for m in sys.modules if m in ('rgn', 'runner') or m.startswith('rgn.')]:   # eski kod bellekte kalmasın
    del sys.modules[_m]
sys.path.insert(0, '/content/RGN/colab'); import runner as R
%cd /content/RGN
COMMIT = !git -C /content/_repo log --oneline -1
print('COMMIT', COMMIT[0])

In [ ]:
# 1 TEST  |  GPU  |  tekrar: GÜVENLİ  (arka planda; sonuç 3 NABIZ: R.nabiz('testler', satir=30))
import torch; assert torch.cuda.is_available(), 'GPU yok: Çalışma zamanı > Çalışma zamanı türünü değiştir'
R.arkaplan('testler', 'python tests/run_tests.py')

In [ ]:
# 2 KOŞU  |  GPU  |  tekrar: SÜRDÜRÜR (aynı ad --resume ile kaldığı yerden; temiz başlatmak için klasörü sil)
# Aşama 0 — SS, D 512, 8 blok, 1 epok.  Ölçüldü (ss_smoke): RGN ~295 ms/adım (~46 dk/epok), transformer ~130 ms/adım (~20 dk).
import torch; assert torch.cuda.is_available(), 'GPU yok'
_free = torch.cuda.mem_get_info()[0] / 2**30; print('boş GPU belleği %.1f GB' % _free); assert _free > 30, 'GPU dolu'
KOSULAR = [
    'configs/ss/ss_d512_rgn.json',              # ana model: çekim + rasyonel MLP, RoPE
    'configs/ss/ss_d512_transformer.json',      # karşılaştırma tabanı: softmax + RoPE + SwiGLU
    'configs/ss/ss_d512_rgn_swiglu.json',       # yalnız MLP değişir
    'configs/ss/ss_d512_softmax_rational.json', # yalnız dikkat değişir
    'configs/ss/ss_d512_rgn_p2.json',           # çekim üssü ||q̂-k̂||⁴
    'configs/ss/ss_d512_rgn_additive.json',     # konum: RoPE yerine β(i-j) paydada
    'configs/ss/ss_d512_rgn_alibi.json',        # konum: çarpımsal β
    'configs/ss/ss_d512_rgn_bf16.json',         # hassasiyet: çekim dikkati bf16
]
R.kuyruk(KOSULAR, name='ss_asama0')

In [ ]:
# 3 NABIZ  |  CPU  |  tekrar: GÜVENLİ
R.nabiz()

In [ ]:
# 4 SONUÇ  |  CPU  |  tekrar: GÜVENLİ   SAYI + METİN
!python scripts/summarize_runs.py
R.sonuc()

In [ ]:
# 5 HIZ  |  GPU  |  tekrar: GÜVENLİ  (arka planda; koşu sürerken çalıştırma — ölçüm bozulur)
R.arkaplan('hiz', 'python scripts/bench_speed.py attention --seq 512 1024 2048 4096 --batch 16')

In [ ]:
# X DURDUR  |  CPU  |  tekrar: GÜVENLİ
# Kuyruğu durdurur: çalışan koşu checkpoint kaydedip çıkar.  Sürdürmek: 2 KOŞU.
R.kapat('ss_asama0')

In [ ]:
# Y DRIVE'DAKİ KAYIT  |  CPU  |  tekrar: GÜVENLİ
from rgn.paths import runs_dir
for _d in sorted(p for p in runs_dir().iterdir() if p.is_dir()):
    _fs = sorted(f.name for f in _d.iterdir())
    print('%-30s %s' % (_d.name, ' '.join(_fs)))

In [ ]:
# Z GPU  |  CPU  |  tekrar: GÜVENLİ
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu --format=csv
# İş yoksa çalışma zamanını kapat (önce 3 NABIZ ile çalışan iş olmadığını gör):
# from google.colab import runtime; runtime.unassign()